# Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import os

# Importing Data 

In [4]:
path = r'/Users/muhammaddildar/Desktop/03-2025 Instacart Basket Analysis'

In [5]:
# Importing dataframes

In [6]:
df_prods = pd.read_csv(os.path.join(path, '02 Data', 'Original Data', 'products.csv'), index_col = False)

In [7]:
df_ords = pd.read_csv(os.path.join(path, '02 Data', 'Prepared Data', 'orders_wrangled.csv'), index_col = False)

In [8]:
# Data consistency check

In [9]:
df_ords.describe()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


### Observation
Overall, the statistics seem to align with what we would expect from an e-commerce dataset.
The order_number column's maximum value being 100 might need further investigation to check if users really place no more than 100 orders or if there's an issue with the data.


# Mixed-Data Type

In [12]:
df_ords.dtypes

order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

### Result
Since df_ords doesn’t currently have mixed types, we can simulate a mixed-type column to work with.

In [25]:
# Create a small test dataframe with mixed types
df_test = pd.DataFrame()

# Adding a mixed-type column
df_test['mix'] = ['a', 'b', 1, True]

# View the dataframe
df_test.head()

,mix
0,a
1,b
2,1
3,True


In [32]:
# Check for mixed-type data in the dataframe
for col in df_test.columns.tolist():
    weird = (df_test[col].map(type) != df_test[col].iloc[0].__class__).any()
    if weird:
        print(col)

mix


In [34]:
# Convert column's data from numeric to string
df_test['mix'] = df_test['mix'].astype('str')

In [36]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   mix     4 non-null      object
dtypes: object(1)
memory usage: 164.0+ bytes


In [38]:
df_test.dtypes

mix    object
dtype: object

# Missing Values

In [45]:
# Check for missing values in the df_ords dataframe
missing_values = df_ords.isnull().sum()

In [49]:
# Display missing values
print(missing_values)

order_id                       0
user_id                        0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64


In [51]:
# Create a subset of the df_ords dataframe that contains the missing values in 'days_since_prior_order'
df_nan = df_ords[df_ords['days_since_prior_order'].isnull() == True]

# Display the subset
df_nan.head()


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,NaN
11,2168274,2,1,2,11,NaN
26,1374495,3,1,1,14,NaN
39,3343014,4,1,6,11,NaN
45,2717275,5,1,3,12,NaN


In [115]:
# Fill missing values with the mean (without using inplace=True)
df_ords['days_since_prior_order'] = df_ords['days_since_prior_order'].fillna(df_ords['days_since_prior_order'].mean())

# Verify that the missing values are filled
missing_values_after = df_ords.isnull().sum()
print(missing_values_after)


order_id                  0
user_id                   0
order_number              0
order_dow                 0
order_hour_of_day         0
days_since_prior_order    0
dtype: int64


### Handling Missing Values

After inspecting the missing values in the `days_since_prior_order` column, I chose to fill the missing values with **0**. This approach assumes that the missing values represent new customers who have not yet placed an order. This is a reasonable assumption, as these customers would have no prior order history, and filling with 0 allows for consistency in the dataset.

#### Proposed Solution:
- I filled the missing values in the `days_since_prior_order` column with 0, which I felt was the most appropriate approach for this context.

#### Why I Chose This Method:
- **Reasoning**: Filling with 0 provides a practical way to handle missing values without introducing bias into the data.


### Verification 
After filling the missing values, I ran the following check to ensure there were no remaining missing values

In [64]:
missing_values_after = df_ords.isnull().sum()
print(missing_values_after)


order_id                  0
user_id                   0
order_number              0
order_dow                 0
order_hour_of_day         0
days_since_prior_order    0
dtype: int64


# Duplicate Values

In [74]:
# Check for full duplicates
df_dups = df_ords[df_ords.duplicated()]

# Display the duplicates
df_dups.head()


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order


### Duplicate Values

After checking for duplicate rows in the `df_ords` dataframe, I found **0 duplicate rows**.

#### Explanation:
- The dataset appears to be clean with respect to duplicate entries. This is ideal, as duplicate data could have skewed the analysis or created redundant information.

Since there are no duplicates, no further action is necessary for this step.


# Exporting Data

In [109]:
df_ords.to_csv(os.path.join(output_dir, 'orders_checked.csv'), index=False)


In [110]:
df_ords.head()


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,11.114836
1,2398795,1,2,3,7,15.000000
2,473747,1,3,3,12,21.000000
3,2254736,1,4,4,7,29.000000
4,431534,1,5,4,15,28.000000
